# Example 7.11.1 / 1: Assignment Formulation

Companion notebook for **7.11.1 Bin Packing and Cutting** in the AMPL lab Documentation, section "Assignment Formulation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Assignment Formulation

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                        # items to pack
set BINS ordered;                 # available identical bins
param size {ITEMS} > 0;           # size of each item
param capacity > 0;               # capacity of every bin
var Assign {ITEMS, BINS} binary;  # 1 if the item goes into the bin
var Use {BINS} binary;            # 1 if the bin is used

minimize BinsUsed: sum {b in BINS} Use[b];  # number of bins used
# each item goes into exactly one bin
subject to PlaceOnce {i in ITEMS}:
    sum {b in BINS} Assign[i,b] = 1;
# contents fit the bin; only a used bin may hold items
subject to BinCapacity {b in BINS}:
    sum {i in ITEMS} size[i] * Assign[i,b] <= capacity * Use[b];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["a", "b", "c", "d", "e", "f", "g"]
ampl.param["size"] = {"a": 7, "b": 6, "c": 5, "d": 4, "e": 3, "f": 3, "g": 2}
ampl.param["capacity"] = 10
ampl.set["BINS"] = [1, 2, 3, 4]

In [5]:
# run.py
ampl.solve(solver="highs")
packed = ampl.var["Assign"].to_dict()
for b in ampl.set["BINS"].to_list():
    print(b, [i for (i, bin), v in packed.items() if bin == b and v > 0.5])

HiGHS 1.14.0: optimal solution; objective 3
14 simplex iterations
1 branching nodes
1 ['b', 'd']
2 []
3 ['c', 'e', 'g']
4 ['a', 'f']


### Pure AMPL: data.dat and model.run

The sizes add up to 30, so `ceil(30 / 10) = 3` bins is a lower bound, and the solver reaches it
with every used bin exactly full; which three of the four candidate bins it uses is arbitrary,
since they are identical.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ITEMS := a b c d e f g;
param size := a 7  b 6  c 5  d 4  e 3  f 3  g 2;
param capacity := 10;
set BINS := 1 2 3 4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
for {b in BINS} {
    printf "%d", b;
    printf {i in ITEMS: Assign[i,b] > 0.5} " %s", i;
    printf "\n";
}

HiGHS 1.14.0: optimal solution; objective 3
14 simplex iterations
1 branching nodes
1 b d
2
3 c e g
4 a f
